# 02 - Main experiment matrix - batch 2

Same protocol as `02_main_exps_1.ipynb`: 45k train / 5k val, seeds 0/42/123, all decisions on
validation, collapsed runs kept and flagged. Runs are saved to the same `task_a/runs/`
directory and appear automatically in `03_final_analysis.ipynb`.

**Experiment numbering:** use `exp_2xx` for this batch; `exp_101`-`exp_109` belong to batch 1.
The same `exp` number across seeds is one configuration; the master table groups by `exp`.

**How to run:** run Setup and Data loaders first. After that, every experiment block is
independent. Long runs (S2, S2c, S5) use `skip_existing=True` and can be resumed by rerunning
the same cell.


## Setup - imports and GPU check

In [1]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from src.config import PROJECT_ROOT, DATA_DIR, RUNS_DIR, SEED, CIFAR100_MEAN, CIFAR100_STD
from src.models import baseline_cnn, baseline_cnn_bn, cnn_3x3, cnn_down
from src.models import (cnn_residual, cnn_wide, cnn_deep, cnn_3stage,
                        cnn_dw, cnn_dw_wide, cnn_groupnorm, cnn_deep_residual,
                        cnn_deep_residual_wide,
                        cnn_deep_residual_d3, cnn_deep_residual_d4, cnn_wide3)   # batch-2 architecture candidates
from src.data import build_datasets, build_loaders, build_loaders_val
from src.engine import train_one_epoch, evaluate, fit
from src.utils import set_seed, save_run, check_gpu, warmup_cosine
from src.experiments import (
    run_experiment,
    run_batch,
    build_experiment_metadata,
    build_architecture_ablation_batch,
)

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

cuda NVIDIA GeForce RTX 4060 Laptop GPU


## Data loaders - no-aug / crop+flip / crop+flip+Cutout

In [2]:
# Shared loaders: no-aug (NOAUG) and augmented (AUG), with the same 45k/5k split.
#
# num_workers: measured 2026-10-16 (cnn_down + crop/flip, batch 128):
# nw=0 was 58.9 ms/batch (20.7 s/epoch); nw=2 and nw=4 both gave 22.8 ms/batch
# (8.0 s/epoch), matching the pure-GPU floor of 22.6 ms. The bottleneck is
# single-process loading, not the GPU. Use 2 rather than 4: same speed, but each
# worker imports torch and keeps a dataset copy, so fewer processes save memory.
# The first batch pays a one-time ~7-13 s worker startup (persistent_workers pays it once).
#
# If Jupyter hangs on the first batch (a Windows spawn issue), set NUM_WORKERS=0;
# everything else stays the same.
NUM_WORKERS = 2

train_loader, val_loader, test_loader = build_loaders_val(batch_size=128, num_workers=NUM_WORKERS)

from torchvision import transforms
aug_train_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])
aug_loader, aug_val, _ = build_loaders_val(batch_size=128, num_workers=NUM_WORKERS, train_tf=aug_train_tf)

# Cutout (RandomErasing): randomly erase a small patch (p=0.5) on top of crop+flip.
# A separate loader: exp_220 differs from exp_212 by this variable only.
erase_train_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
    transforms.RandomErasing(p=0.5, scale=(0.02, 0.2), ratio=(0.3, 3.3), value=0),
])
erase_loader, erase_val, _ = build_loaders_val(batch_size=128, num_workers=NUM_WORKERS, train_tf=erase_train_tf)

# Build the three loader entries; training happens in the later cells.
NOAUG = (train_loader, val_loader)
AUG = (aug_loader, aug_val)
ERASE = (erase_loader, erase_val)

print(f"no-aug: train {len(train_loader)} batches | val {len(val_loader)} batches")
print(f"aug:    train {len(aug_loader)} batches | val {len(aug_val)} batches")
print(f"erase:  train {len(erase_loader)} batches | val {len(erase_val)} batches")

# Comparability note: with num_workers>0, augmented runs use each worker's RNG,
# so they are not bitwise comparable to batch-1 augmented runs (nw=0), although
# they are statistically unbiased. No-augmentation experiments are unaffected:
# transforms are deterministic and the shuffle order still comes from the main
# process RNG.

no-aug: train 351 batches | val 40 batches
aug:    train 351 batches | val 40 batches
erase:  train 351 batches | val 40 batches


## Shared infrastructure

All batch runners live in `src/` and are shared with batch 1: `run_experiment` (multi-seed
training, plots, collapse/slow-state warnings), `build_experiment_metadata` (metadata only),
`check_gpu` and `warmup_cosine` (utils), and `fit` (single training run). Each experiment cell
only depends on the imports and loaders above.


## S1 - Architecture ablation (20 epochs, augmentation on)

Why 20 epochs with augmentation: `cnn_down` without augmentation reaches 96% train accuracy
within 10 epochs (exp_109), so the architecture is already overfitting. Screening architectures
without augmentation would systematically punish larger models. Screening therefore uses the
same regularization regime as the final long runs, at a cost of ~25 s/epoch.

**Anchor:** `exp_200` = `cnn_down` at 20 epochs with 3 seeds. Every other row is measured
relative to it. Each candidate changes exactly one thing and uses 1 seed; winners are
confirmed with 3 seeds in S1b.

| exp | Change | Factor | Params |
|---|---|---|---|
| 200 | base (anchor, 3 seeds) | - | 2.35M |
| 201 | + residual connection (same shape) | 7b | 2.46M |
| 202 | width x2 | 7f | 9.35M |
| 203 | depth x2 (2 -> 4 3x3 per stage) | 7f | 5.09M |
| 204 | drop the 4px stage (4 -> 3 stages) | 7d | 1.17M |
| 205 | depthwise-separable (same shape) | 7c | 0.30M |
| 206 | depthwise + wider channels (~same params) | 7c | 2.45M |
| 207 | BatchNorm -> GroupNorm(8) | 7e | 2.35M |

> These numbers are only comparable inside this block (20 epochs, augmentation on).
> Do not mix them with the 10-epoch batch-1 numbers.

The batch is built by `build_architecture_ablation_batch()`. The preview cell only prints the
plan; the next cell starts training.


### S1 preview - configuration and parameter counts

In [ ]:
# The S1 batch is built by src/experiments.py.
# This cell only previews the configuration; the next cell trains.
S1_EPOCHS = 20
S1_BATCH = build_architecture_ablation_batch()

print(f"{len(S1_BATCH)} configs: 1 three-seed anchor + 7 one-seed candidates | epochs={S1_EPOCHS}")
for c in S1_BATCH:
    n = sum(p.numel() for p in c["model"]().parameters())
    print(f"  {c['exp']}_{c['name']:<10} {n:>10,} params   seeds={c['seeds']}")

### S1 run - 8 configurations (~1.5-2 h)

In [4]:
# Run S1 (~1.5-2 h; do not let the machine sleep).
# Depends only on Setup + Data loaders; rerun the cell if training is interrupted.
# skip_existing skips configurations that are already saved.
S1_EPOCHS = 20
s1_summary = run_batch(build_architecture_ablation_batch(), loaders=AUG, epochs=S1_EPOCHS)

epoch  1/20 | train 4.003/0.108 | test 3.616/0.153 | 21s
epoch  2/20 | train 3.382/0.184 | test 3.004/0.245 | 21s
epoch  3/20 | train 2.813/0.282 | test 2.866/0.291 | 21s
epoch  4/20 | train 2.354/0.379 | test 2.311/0.399 | 22s
epoch  5/20 | train 2.040/0.444 | test 2.117/0.441 | 22s
epoch  6/20 | train 1.817/0.495 | test 1.856/0.497 | 22s
epoch  7/20 | train 1.637/0.541 | test 1.817/0.501 | 22s
epoch  8/20 | train 1.485/0.580 | test 1.711/0.541 | 25s
epoch  9/20 | train 1.359/0.611 | test 1.619/0.573 | 24s
epoch 10/20 | train 1.237/0.641 | test 1.624/0.567 | 25s
epoch 11/20 | train 1.138/0.669 | test 1.457/0.608 | 25s
epoch 12/20 | train 1.037/0.696 | test 1.403/0.627 | 24s
epoch 13/20 | train 0.946/0.721 | test 1.315/0.641 | 24s
epoch 14/20 | train 0.867/0.743 | test 1.322/0.642 | 24s
epoch 15/20 | train 0.787/0.765 | test 1.268/0.658 | 24s
epoch 16/20 | train 0.730/0.786 | test 1.228/0.664 | 24s
epoch 17/20 | train 0.673/0.803 | test 1.210/0.677 | 24s
epoch 18/20 | train 0.627/0.816

## S1b - Confirm the winner (3 seeds) + exp_208 (depth + residual)

- **exp_202 confirmation:** width x2 produced +3.79 with 1 seed in S1; confirm it with 3 seeds
  before reporting. The anchor exp_200 already uses 3 seeds, so the delta is clean. The
  confirmation uses a new run name `wide_conf`; the original 1-seed `wide` run stays in
  `runs/`, so both `tier=screen` and `tier=confirm` remain visible in the master table.
- **exp_208 = depth x2 + residuals:** residuals alone (exp_201, -0.02) and depth alone
  (exp_203, -3.11) were both negative. This tests whether residuals unlock depth.

Expected: ~40 min with `NUM_WORKERS=2`.


### S1b run - confirmation + exp_208 (~40 min)

In [3]:
# S1b + exp_208 (~40 min). Depends only on Setup + Data loaders.
# Rerun the cell if interrupted.
S1B_EPOCHS = 20
S1B_BATCH = [
    {"exp": "exp_202", "name": "wide_conf", "model": cnn_wide, "seeds": (0, 42, 123),
     "extra": build_experiment_metadata("down+wide", "confirm")},
    {"exp": "exp_208", "name": "deep_residual", "model": cnn_deep_residual, "seeds": (0,),
     "extra": build_experiment_metadata("down+deep+residual", "screen")},
]
run_batch(S1B_BATCH, loaders=AUG, epochs=S1B_EPOCHS)

epoch  1/20 | train 3.749/0.139 | test 3.253/0.215 | 36s
epoch  2/20 | train 3.124/0.229 | test 2.887/0.271 | 25s
epoch  3/20 | train 2.621/0.317 | test 2.708/0.324 | 25s
epoch  4/20 | train 2.167/0.416 | test 2.119/0.440 | 25s
epoch  5/20 | train 1.870/0.485 | test 2.164/0.445 | 25s
epoch  6/20 | train 1.649/0.538 | test 1.985/0.477 | 25s
epoch  7/20 | train 1.482/0.577 | test 1.675/0.535 | 25s
epoch  8/20 | train 1.321/0.617 | test 1.676/0.559 | 25s
epoch  9/20 | train 1.188/0.653 | test 1.544/0.588 | 26s
epoch 10/20 | train 1.061/0.688 | test 1.515/0.601 | 25s
epoch 11/20 | train 0.944/0.718 | test 1.426/0.622 | 26s
epoch 12/20 | train 0.829/0.753 | test 1.361/0.646 | 26s
epoch 13/20 | train 0.727/0.781 | test 1.310/0.658 | 26s
epoch 14/20 | train 0.626/0.812 | test 1.293/0.668 | 25s
epoch 15/20 | train 0.529/0.844 | test 1.220/0.679 | 25s
epoch 16/20 | train 0.454/0.866 | test 1.192/0.690 | 25s
epoch 17/20 | train 0.388/0.889 | test 1.169/0.693 | 25s
epoch 18/20 | train 0.344/0.907

[('exp_202_wide_conf', [0.7038, 0.7042, 0.704], 20, 3),
 ('exp_208_deep_residual', [0.6908], 20, 1)]

## S1c - 3-seed confirmation of exp_208 (optional, ~21 min)

exp_208 (depth x2 + residuals) is the second positive result in S1, but it currently has only
1 seed and +1.45 pt is close to the noise floor (~1 pt). Three seeds turn it from directional
evidence into a reportable conclusion. Use the new run name `deep_residual_conf`; keep the
original 1-seed run. `tier=confirm`, so both rows remain in the master table.

This is optional for S2, but a cheap 21-minute safeguard.


### S1c run - 3-seed confirmation of exp_208 (~21 min)

In [4]:
# 3-seed confirmation of exp_208 (~21 min, ~7 min per seed). Depends only on Setup + Data loaders.
S1C_EPOCHS = 20
S1C_BATCH = [
    {"exp": "exp_208", "name": "deep_residual_conf", "model": cnn_deep_residual, "seeds": (0, 42, 123),
     "extra": build_experiment_metadata("down+deep+residual", "confirm")},
]
run_batch(S1C_BATCH, loaders=AUG, epochs=S1C_EPOCHS)


epoch  1/20 | train 3.783/0.125 | test 3.413/0.178 | 21s
epoch  2/20 | train 3.220/0.209 | test 2.990/0.246 | 21s
epoch  3/20 | train 2.705/0.300 | test 2.573/0.332 | 21s
epoch  4/20 | train 2.224/0.403 | test 2.357/0.391 | 21s
epoch  5/20 | train 1.920/0.471 | test 1.971/0.464 | 21s
epoch  6/20 | train 1.705/0.521 | test 2.064/0.456 | 21s
epoch  7/20 | train 1.538/0.562 | test 1.686/0.541 | 21s
epoch  8/20 | train 1.396/0.599 | test 1.622/0.549 | 21s
epoch  9/20 | train 1.260/0.633 | test 1.511/0.590 | 21s
epoch 10/20 | train 1.144/0.663 | test 1.475/0.593 | 20s
epoch 11/20 | train 1.029/0.696 | test 1.479/0.598 | 20s
epoch 12/20 | train 0.928/0.722 | test 1.346/0.635 | 21s
epoch 13/20 | train 0.824/0.749 | test 1.280/0.649 | 21s
epoch 14/20 | train 0.722/0.781 | test 1.264/0.656 | 21s
epoch 15/20 | train 0.635/0.806 | test 1.228/0.672 | 21s
epoch 16/20 | train 0.553/0.834 | test 1.175/0.683 | 21s
epoch 17/20 | train 0.484/0.855 | test 1.160/0.687 | 21s
epoch 18/20 | train 0.434/0.871

[('exp_208_deep_residual_conf', [0.6982, 0.6862, 0.69], 20, 3)]

## S2 - Overnight long runs (60 epochs, 3 seeds, 9 configurations) - core of the report

Order = priority: the first four runs already answer the two hard report questions (when
overfitting starts, and whether augmentation wins at a long budget). With `skip_existing=True`,
rerunning this cell resumes from the last completed configuration.

| exp | Configuration | Question | Estimate |
|---|---|---|---|
| 211 | `cnn_wide`, 60 ep, no augmentation | when overfitting starts | 1.3 h |
| 212 | `cnn_wide`, 60 ep, crop+flip | does augmentation win at a long budget? | 1.3 h |
| 213 | `cnn_deep_residual`, 60 ep, no augmentation | does depth+residual grow or vanish? | 1.1 h |
| 214 | `cnn_deep_residual`, 60 ep, crop+flip | same, plus this configuration's augmentation sign | 1.1 h |
| 216 | `cnn_deep_residual_wide`, 60 ep, aug | do width and depth+residual add? | 3.2 h |
| 217 | `cnn_deep_residual_d3`, 60 ep, aug | depth ladder point 3 (7.9M) | 1.6 h |
| 218 | `cnn_deep_residual_d4`, 60 ep, aug | depth ladder point 4 (10.7M) | 2.2 h |
| 219 | `cnn_wide3`, 60 ep, aug | width ladder point 3 (21.0M) | 2.6 h |
| 220 | `cnn_wide` + Cutout, 60 ep, aug | third augmentation family | 1.3 h |

Total: ~15.7 h (more in the GPU slow state; rerun the cell to resume).

Design notes:
- 211-214 form a 2x2: the augmentation sign is measured on two architectures.
- Depth ladder 1 -> 2 -> 3 -> 4 residual blocks; width ladder 1x -> 2x -> 3x. Both ladders use
  augmentation and are internally comparable.
- 216 combines the two positive knobs from S1.
- 220 differs from 212 only by Cutout (separate loaders in the data-loaders cell).
- Every run records an EMA curve (`ema_decay=0.999`); EMA only affects logging.
- The epoch-ladder plot is just the per-epoch validation curves of these nine runs.

Before starting: confirm the laptop performance mode and run the short GPU check.
Do not edit this notebook while training is running.


### S2 run - 9 configurations x 3 seeds x 60 epochs (~15.7 h)

In [ ]:
# S2 overnight runs: 9 configurations x 3 seeds x 60 epochs. Depends only on Setup + Data loaders.
# Order = priority; rerun this cell after an interruption and skip_existing resumes.
# Every run records an EMA curve (run_batch default ema_decay=0.999; set "ema_decay": None to disable).
print("9 configs x 3 seeds x 60 epochs | ~15.7 h (slower in the GPU slow state; rerun to resume)")
S2_EPOCHS = 60

S2_BATCH = [
    # --- 2x2 core: overfitting onset + augmentation verdict ---
    {"exp": "exp_211", "name": "wide_noaug",     "model": cnn_wide,                "seeds": (0, 42, 123),
     "extra": build_experiment_metadata("down+wide",              "confirm", aug="none"),      "loaders": NOAUG},
    {"exp": "exp_212", "name": "wide_aug",       "model": cnn_wide,                "seeds": (0, 42, 123),
     "extra": build_experiment_metadata("down+wide",              "confirm", aug="crop+flip"), "loaders": AUG},
    {"exp": "exp_213", "name": "deepres_noaug",  "model": cnn_deep_residual,       "seeds": (0, 42, 123),
     "extra": build_experiment_metadata("down+deep+residual",     "confirm", aug="none"),      "loaders": NOAUG},
    {"exp": "exp_214", "name": "deepres_aug",    "model": cnn_deep_residual,       "seeds": (0, 42, 123),
     "extra": build_experiment_metadata("down+deep+residual",     "confirm", aug="crop+flip"), "loaders": AUG},
    # --- Combination: do the two wins add? ---
    {"exp": "exp_216", "name": "dr_wide_aug",    "model": cnn_deep_residual_wide,  "seeds": (0, 42, 123),
     "extra": build_experiment_metadata("down+deep+residual+wide", "confirm", aug="crop+flip"), "loaders": AUG},
    # --- Depth ladder (all with residuals) ---
    {"exp": "exp_217", "name": "dr_d3_aug",      "model": cnn_deep_residual_d3,    "seeds": (0, 42, 123),
     "extra": build_experiment_metadata("down+deep3+residual",    "confirm", aug="crop+flip"), "loaders": AUG},
    {"exp": "exp_218", "name": "dr_d4_aug",      "model": cnn_deep_residual_d4,    "seeds": (0, 42, 123),
     "extra": build_experiment_metadata("down+deep4+residual",    "confirm", aug="crop+flip"), "loaders": AUG},
    # --- Width ladder ---
    {"exp": "exp_219", "name": "wide3_aug",      "model": cnn_wide3,               "seeds": (0, 42, 123),
     "extra": build_experiment_metadata("down+wide3",             "confirm", aug="crop+flip"), "loaders": AUG},
    # --- Augmentation family: Cutout (the only change vs 212) ---
    {"exp": "exp_220", "name": "wide_cutout",    "model": cnn_wide,                "seeds": (0, 42, 123),
     "extra": build_experiment_metadata("down+wide",              "confirm", aug="crop+flip+erase"), "loaders": ERASE},
]
run_batch(S2_BATCH, epochs=S2_EPOCHS)


## S2c - 60-epoch baseline anchor (`cnn_down`, ~1.2 h)

Both capacity ladders need the same missing point: width x1 / depth 1 block, i.e. `cnn_down`
at 60 epochs. This run completes both ladders and provides a same-budget reference for every
delta. It also lets us test additivity: `216 ~ base + (wide - base) + (dr - base)`.

The run name is `base_down60`, distinct from the 20-epoch `exp_200_base_down`.


### S2c run - 60-epoch baseline anchor (~1.2 h)

In [3]:
# S2c: 60-epoch baseline anchor (~1.2 h, 3 seeds). Depends only on Setup + Data loaders.
S2C_EPOCHS = 60
S2C_BATCH = [
    {"exp": "exp_221", "name": "base_down60", "model": cnn_down, "seeds": (0, 42, 123),
     "extra": build_experiment_metadata("downsampled", "anchor", aug="crop+flip"), "loaders": AUG},
]
run_batch(S2C_BATCH, epochs=S2C_EPOCHS)


epoch  1/60 | train 4.002/0.108 | test 3.616/0.157 | ema 4.605/0.011 | 23s
epoch  2/60 | train 3.353/0.190 | test 3.049/0.239 | ema 4.598/0.009 | 9s
epoch  3/60 | train 2.800/0.282 | test 2.681/0.316 | ema 4.523/0.020 | 9s
epoch  4/60 | train 2.340/0.378 | test 2.284/0.393 | ema 4.378/0.029 | 9s
epoch  5/60 | train 2.043/0.444 | test 2.098/0.443 | ema 4.001/0.072 | 9s
epoch  6/60 | train 1.820/0.496 | test 2.012/0.469 | ema 3.402/0.170 | 9s
epoch  7/60 | train 1.649/0.539 | test 1.804/0.509 | ema 2.727/0.306 | 9s
epoch  8/60 | train 1.509/0.572 | test 1.680/0.546 | ema 2.218/0.418 | 9s
epoch  9/60 | train 1.395/0.601 | test 1.645/0.555 | ema 1.855/0.500 | 9s
epoch 10/60 | train 1.294/0.625 | test 1.666/0.550 | ema 1.621/0.551 | 9s
epoch 11/60 | train 1.216/0.647 | test 1.569/0.580 | ema 1.462/0.597 | 9s
epoch 12/60 | train 1.130/0.670 | test 1.627/0.575 | ema 1.350/0.627 | 9s
epoch 13/60 | train 1.056/0.688 | test 1.418/0.615 | ema 1.277/0.647 | 9s
epoch 14/60 | train 0.985/0.706 | tes

[('exp_221_base_down60', [0.692, 0.6916, 0.6984], 60, 3)]

## S4 - Evaluation and ensembling (no training, ~10 min)

Three evaluations reuse the saved `model.pt` files:

1. **top-5 backfill:** scan `runs/`, recompute top-1/top-5 for every final model on the 5k
   validation split, and cross-check top-1 against the last row of `metrics.csv`. The result
   is written to `runs/topk_summary.csv`.
2. **TTA:** average the probabilities of the original and horizontally flipped views.
3. **3-seed ensemble:** average the probabilities of the three seeds for one configuration.

Only 45k/5k runs are included (`eval_set == "val5k"`); the 50k models in `runs_final/` are
not part of this section.

> **Checkpoint note:** the exploratory `model.pt` files under `task_a/runs/` were deleted
> after the experiments to save ~1.9 GB; the single exception is the collapse evidence
> `runs/exp_002_adam_45k/model.pt`. S4a/S4b are kept for reproducibility but require
> retraining if you want to rerun them; saved results remain in `runs/topk_summary.csv`
> and `runs/*/metrics.csv`. The final models in `runs_final/` are kept for S6d/S6e.


### S4a - top-5 backfill (archived; requires checkpoints)

In [3]:
# S4a: top-5 backfill (evaluation only, ~5-10 min). Writes runs/topk_summary.csv.
import json
import csv
from src.data import build_loaders_val
from src.engine import evaluate_topk
from src.models import (baseline_cnn, baseline_cnn_bn, cnn_3x3, cnn_down, cnn_residual, cnn_wide,
                        cnn_deep, cnn_3stage, cnn_dw, cnn_dw_wide, cnn_groupnorm, cnn_deep_residual,
                        cnn_deep_residual_wide, cnn_deep_residual_d3, cnn_deep_residual_d4, cnn_wide3)

# Map the arch string in config.json to a model factory (the descriptive field written by the batch experiments).
ARCH2MODEL = {
    "baseline": baseline_cnn, "baseline+BN": baseline_cnn_bn, "2x3x3 stack": cnn_3x3,
    "downsampled": cnn_down, "down+residual": cnn_residual, "down+wide": cnn_wide,
    "down+deep": cnn_deep, "3stage": cnn_3stage, "down+dw": cnn_dw, "down+dw+wide": cnn_dw_wide,
    "down+gn": cnn_groupnorm, "down+deep+residual": cnn_deep_residual,
    "down+deep+residual+wide": cnn_deep_residual_wide,
    "down+deep3+residual": cnn_deep_residual_d3, "down+deep4+residual": cnn_deep_residual_d4,
    "down+wide3": cnn_wide3,
}
LEGACY = {"exp_000_baseline", "exp_001_adam", "exp_002_adam_45k"}   # same as 03_final_analysis

_, val_loader, _ = build_loaders_val(batch_size=128, num_workers=NUM_WORKERS)
crit = torch.nn.CrossEntropyLoss()

rows, skipped = [], []
for d in sorted(RUNS_DIR.iterdir()):
    if not d.is_dir() or not (d / "model.pt").exists():
        continue
    cfg = json.loads((d / "config.json").read_text(encoding="utf-8"))
    if d.name in LEGACY or cfg.get("eval_set", "val5k") != "val5k":
        skipped.append(d.name)
        continue
    factory = ARCH2MODEL.get(cfg.get("arch"))
    if factory is None:
        skipped.append(d.name)
        print("!! unknown arch:", cfg.get("arch"), d.name)
        continue

    model = factory().to(device)
    model.load_state_dict(torch.load(d / "model.pt", map_location=device, weights_only=True))
    _, t1, t5 = evaluate_topk(model, val_loader, crit, device)

    last = float(list(csv.DictReader(open(d / "metrics.csv", newline="", encoding="utf-8")))[-1]["test_acc"])
    flag = "" if abs(last - t1) < 1e-6 else "   <-- !! differs from metrics.csv (%.4f)" % last

    rows.append({"run": d.name, "exp": cfg.get("exp"), "arch": cfg.get("arch"), "tier": cfg.get("tier"),
                 "aug": cfg.get("aug"), "epochs": cfg.get("epochs"), "seed": cfg.get("seed"),
                 "val_top1": round(t1, 4), "val_top5": round(t5, 4)})
    print("%-36s top1=%.4f top5=%.4f%s" % (d.name, t1, t5, flag))

out = RUNS_DIR / "topk_summary.csv"
with open(out, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader(); w.writerows(rows)
print()
print("%d runs -> %s | skipped %d: %s" % (len(rows), out, len(skipped), skipped))


exp_101_baseline_sgd_s0              top1=0.1166 top5=0.3186
exp_101_baseline_sgd_s123            top1=0.1108 top5=0.3222
exp_101_baseline_sgd_s42             top1=0.1126 top5=0.3178
exp_102_adam_s0                      top1=0.2794 top5=0.5774
exp_102_adam_s123                    top1=0.3452 top5=0.6466
exp_102_adam_s42                     top1=0.0076 top5=0.0392
exp_103_bn_s0                        top1=0.4746 top5=0.7756
exp_103_bn_s123                      top1=0.4702 top5=0.7702
exp_103_bn_s42                       top1=0.4704 top5=0.7640
exp_104_lrsched_s0                   top1=0.5420 top5=0.8282
exp_104_lrsched_s123                 top1=0.5398 top5=0.8294
exp_104_lrsched_s42                  top1=0.5410 top5=0.8254
exp_105_aug_s0                       top1=0.4974 top5=0.7994
exp_105_aug_s123                     top1=0.5010 top5=0.7968
exp_105_aug_s42                      top1=0.4994 top5=0.8034
exp_106_arch_3x3_s0                  top1=0.5374 top5=0.8366
exp_106_arch_3x3_s123   

### S4b - TTA and 3-seed ensemble for exp_216 (archived; requires checkpoints)

In [3]:
# S4b: TTA and 3-seed ensemble for the champion (exp_216), val5k, ~1 min; reuses model.pt, no retraining.
from src.data import build_loaders_val
from src.engine import evaluate_topk, ensemble_eval

_, val_loader, _ = build_loaders_val(batch_size=128, num_workers=NUM_WORKERS)
crit = torch.nn.CrossEntropyLoss()

models216 = []
for s in (0, 42, 123):
    m = cnn_deep_residual_wide().to(device)
    m.load_state_dict(torch.load(RUNS_DIR / ("exp_216_dr_wide_aug_s%d" % s) / "model.pt",
                                 map_location=device, weights_only=True))
    models216.append(m)
    _, t1, t5 = evaluate_topk(m, val_loader, crit, device)
    print("seed %-3d              top1=%.4f top5=%.4f" % (s, t1, t5))

for label, ms, tta in [("single seed0  raw", models216[:1], False),
                       ("single seed0  TTA", models216[:1], True),
                       ("3-seed ens    raw", models216,    False),
                       ("3-seed ens    TTA", models216,    True)]:
    t1, t5 = ensemble_eval(ms, val_loader, device, tta=tta)
    print("%-20s top1=%.4f top5=%.4f" % (label, t1, t5))
print()
print("Note: these are val5k numbers; the headline is S5's test10k result (evaluated once).")


seed 0                top1=0.7420 top5=0.9298
seed 42               top1=0.7392 top5=0.9284
seed 123              top1=0.7428 top5=0.9290
single seed0  raw    top1=0.7420 top5=0.9298
single seed0  TTA    top1=0.7484 top5=0.9322
3-seed ens    raw    top1=0.7626 top5=0.9380
3-seed ens    TTA    top1=0.7634 top5=0.9408

Note: these are val5k numbers; the headline is S5's test10k result (evaluated once).


## S5 - Final delivery: 50k full retrain, test evaluated once

- Champion recipe (exp_216: `cnn_deep_residual_wide` + crop+flip + 60 epochs), trained on the
  full 50k (the 5k validation split now participates in training).
- No held-out validation set: each epoch logs a monitor curve on a 2k training subset (not
  test; see `src/data.py:build_train_test_loaders`).
- Test is evaluated once after training, per seed, plus TTA and a 3-seed ensemble. Every number
  is reported; nothing is selected after seeing test.
- Artifacts go to `runs_final/`, separate from the 45k/5k protocol in `runs/`.
- Cost: 50k is 11% larger than 45k, ~70 s/epoch -> ~72 min/seed x 3 = ~3.6 h.

After seeing test, do not change the model; that would be test-set tuning.


### S5 run - 50k retrain + one test evaluation (~3.6 h)

In [4]:
# S5: full 50k retrain + one test evaluation (~3.6 h). Depends on Setup + Data loaders.
from src.data import build_train_test_loaders
from src.experiments import run_final_retrain

FIN_EPOCHS = 60
fin_train, fin_test, fin_monitor = build_train_test_loaders(
    batch_size=128, num_workers=NUM_WORKERS, train_tf=aug_train_tf, monitor_size=2000)

final_summary = run_final_retrain(
    "exp_230", "final_dr_wide", cnn_deep_residual_wide,
    train_loader=fin_train, test_loader=fin_test, monitor_loader=fin_monitor,
    extra=build_experiment_metadata("down+deep+residual+wide", "final", aug="crop+flip"),
    seeds=(0, 42, 123), epochs=FIN_EPOCHS, batch_size=128, device=device, tta=True)


epoch  1/60 | train 3.542/0.160 | test 2.980/0.268 | 80s
epoch  2/60 | train 3.000/0.247 | test 2.651/0.327 | 66s
epoch  3/60 | train 2.469/0.352 | test 2.509/0.359 | 66s
epoch  4/60 | train 1.990/0.455 | test 2.132/0.423 | 66s
epoch  5/60 | train 1.716/0.517 | test 1.635/0.551 | 66s
epoch  6/60 | train 1.528/0.562 | test 1.412/0.595 | 66s
epoch  7/60 | train 1.370/0.605 | test 1.418/0.593 | 66s
epoch  8/60 | train 1.244/0.636 | test 1.570/0.578 | 66s
epoch  9/60 | train 1.133/0.665 | test 1.387/0.599 | 67s
epoch 10/60 | train 1.035/0.690 | test 0.948/0.711 | 66s
epoch 11/60 | train 0.933/0.718 | test 0.956/0.700 | 66s
epoch 12/60 | train 0.857/0.738 | test 0.831/0.750 | 66s
epoch 13/60 | train 0.780/0.760 | test 0.748/0.759 | 66s
epoch 14/60 | train 0.706/0.780 | test 0.710/0.774 | 66s
epoch 15/60 | train 0.639/0.799 | test 0.600/0.803 | 66s
epoch 16/60 | train 0.568/0.820 | test 0.479/0.850 | 66s
epoch 17/60 | train 0.503/0.838 | test 0.492/0.847 | 67s
epoch 18/60 | train 0.440/0.858